# AIS Distance Prediction Training

Train an image-based position predictor from the Hugging Face `vision_offset_dataset`.
Data is expected under `ws_aic/data`, and checkpoints are saved under `ws_aic/model`.

In [1]:
from pathlib import Path
import sys

PROJECT_DIR = Path('/home/vsc/LLM_TUNE/AIC_Sejong/ws_aic/src/ais/ais_distance_prediction')
if str(PROJECT_DIR) not in sys.path:
    sys.path.insert(0, str(PROJECT_DIR))

PROJECT_DIR

PosixPath('/home/vsc/LLM_TUNE/AIC_Sejong/ws_aic/src/ais/ais_distance_prediction')

In [2]:
import torch
from torch.utils.data import DataLoader
from torchvision import transforms

from model import (
    DEFAULT_DATASET_ROOT,
    DEFAULT_WEIGHT_ROOT,
    VisionOffsetDataset,
    build_resnet_position_model,
    compute_target_stats,
    download_vision_offset_dataset,
    evaluate,
    fit,
    load_samples,
    seed_everything,
    split_samples_by_episode,
)

seed_everything(42)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
device

device(type='cuda')

In [3]:
DATASET_ROOT = DEFAULT_DATASET_ROOT
WEIGHT_ROOT = DEFAULT_WEIGHT_ROOT

CAMERAS = ('center',)  # Use ('left', 'center', 'right') for multi-view training.
TARGET_KEYS = ('x_mm', 'y_mm', 'z_mm')
IMAGE_SIZE = 224
BATCH_SIZE = 16
NUM_WORKERS = 4
EPOCHS = 20
LR = 3e-4
WEIGHT_DECAY = 1e-4
BACKBONE = 'resnet50'
PRETRAINED = True
RUN_NAME = f"vision_offset_{BACKBONE}_{'_'.join(CAMERAS)}"

DATASET_ROOT, WEIGHT_ROOT / RUN_NAME

(PosixPath('/home/vsc/LLM_TUNE/AIC_Sejong/ws_aic/src/data/aic-entrance-dataset/v1.1/vision_offset_dataset'),
 PosixPath('/home/vsc/LLM_TUNE/AIC_Sejong/ws_aic/model/ais_distance_prediction/vision_offset_resnet50_center'))

In [4]:
# DATASET_ROOT = download_vision_offset_dataset(max_workers=8)

samples = load_samples(DATASET_ROOT)
train_samples, val_samples, test_samples = split_samples_by_episode(
    samples,
    val_ratio=0.2,
    test_ratio=0.0,
    seed=42,
)
target_stats = compute_target_stats(train_samples, TARGET_KEYS)

print(f"samples: train={len(train_samples)} val={len(val_samples)} test={len(test_samples)}")
print('target mean:', target_stats['mean'].tolist())
print('target std :', target_stats['std'].tolist())

samples: train=3520 val=960 test=0
target mean: [1.4557099342346191, 0.07912462204694748, -19.024002075195312]
target std : [5.113223075866699, 2.4719314575195312, 18.063444137573242]


In [5]:
train_transform = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.ColorJitter(brightness=0.15, contrast=0.15, saturation=0.08, hue=0.02),
    transforms.ToTensor(),
    transforms.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
])

eval_transform = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
])

In [6]:
train_dataset = VisionOffsetDataset(
    DATASET_ROOT,
    samples=train_samples,
    cameras=CAMERAS,
    target_keys=TARGET_KEYS,
    transform=train_transform,
    target_mean=target_stats['mean'],
    target_std=target_stats['std'],
)
val_dataset = VisionOffsetDataset(
    DATASET_ROOT,
    samples=val_samples,
    cameras=CAMERAS,
    target_keys=TARGET_KEYS,
    transform=eval_transform,
    target_mean=target_stats['mean'],
    target_std=target_stats['std'],
)

pin_memory = device.type == 'cuda'
train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    pin_memory=pin_memory,
    drop_last=False,
)
val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=pin_memory,
    drop_last=False,
)

batch = next(iter(train_loader))
batch['image'].shape, batch['target'].shape, batch['raw_target'][:3]

/home/vsc/LLM_TUNE/structure-stability/venv/venv-stability/lib/python3.10/site-packages/torch/cuda/__init__.py:435: UserWarning: 
    Found GPU0 NVIDIA GB10 which is of cuda capability 12.1.
    Minimum and Maximum cuda capability supported by this version of PyTorch is
    (8.0) - (12.0)
    
  queued_call()


(torch.Size([16, 3, 224, 224]),
 torch.Size([16, 3]),
 tensor([[  1.7391,   3.4794, -14.7074],
         [  1.6696,   1.6502, -16.3671],
         [  1.5583,   2.3538, -15.7988]]))

In [7]:
model = build_resnet_position_model(
    backbone_name=BACKBONE,
    pretrained=PRETRAINED,
    output_dim=len(TARGET_KEYS),
    hidden_dim=256,
    dropout=0.1,
    aggregation='mean',
)
model.to(device)

optimizer = torch.optim.AdamW(
    (p for p in model.parameters() if p.requires_grad),
    lr=LR,
    weight_decay=WEIGHT_DECAY,
)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode='min',
    factor=0.5,
    patience=3,
)

sum(p.numel() for p in model.parameters() if p.requires_grad)

Downloading: "https://download.pytorch.org/models/resnet50-11ad3fa6.pth" to /home/vsc/.cache/torch/hub/checkpoints/resnet50-11ad3fa6.pth


100%|██████████| 97.8M/97.8M [00:11<00:00, 8.76MB/s]


24065859

In [8]:
config = {
    'dataset_root': str(DATASET_ROOT),
    'cameras': CAMERAS,
    'target_keys': TARGET_KEYS,
    'target_mean': target_stats['mean'].tolist(),
    'target_std': target_stats['std'].tolist(),
    'image_size': IMAGE_SIZE,
    'batch_size': BATCH_SIZE,
    'epochs': EPOCHS,
    'lr': LR,
    'weight_decay': WEIGHT_DECAY,
    'backbone': BACKBONE,
    'pretrained': PRETRAINED,
}

history = fit(
    model,
    train_loader,
    val_loader,
    optimizer,
    device,
    epochs=EPOCHS,
    weight_dir=WEIGHT_ROOT,
    run_name=RUN_NAME,
    scheduler=scheduler,
    target_mean=target_stats['mean'],
    target_std=target_stats['std'],
    config=config,
)
history[-1]

epochs:   0%|          | 0/20 [00:00<?, ?it/s]

train 1/20:   0%|          | 0/220 [00:00<?, ?it/s]

val 1/20:   0%|          | 0/60 [00:00<?, ?it/s]

epoch=001 train_loss=0.0971 val_loss=0.0851 val_mae=1.864 val_euclidean=3.817


train 2/20:   0%|          | 0/220 [00:00<?, ?it/s]

...

epoch=019 train_loss=0.0066 val_loss=0.0907 val_mae=1.643 val_euclidean=3.233


train 20/20:   0%|          | 0/220 [00:00<?, ?it/s]

val 20/20:   0%|          | 0/60 [00:00<?, ?it/s]

epoch=020 train_loss=0.0069 val_loss=0.0971 val_mae=1.682 val_euclidean=3.319


{'epoch': 20.0,
 'train_loss': 0.006868065230082721,
 'val_loss': 0.09712736277530591,
 'val_mae': 1.6816662549972534,
 'val_rmse': 2.118018865585327,
 'val_euclidean': 3.319140672683716}

In [9]:
val_metrics = evaluate(
    model,
    val_loader,
    device,
    target_mean=target_stats['mean'],
    target_std=target_stats['std'],
)
val_metrics

Exception ignored in: 

val:   0%|          | 0/60 [00:00<?, ?it/s]

Exception ignored in: Exception ignored in: <function _MultiProcessingDataLoaderIter.__del__ at 0xfdede9f81510><function _MultiProcessingDataLoaderIter.__del__ at 0xfdede9f81510><function _MultiProcessingDataLoaderIter.__del__ at 0xfdede9f81510>
Traceback (most recent call last):


  File "/home/vsc/LLM_TUNE/structure-stability/venv/venv-stability/lib/python3.10/site-packages/torch/utils/data/dataloader.py", line 1707, in __del__
Traceback (most recent call last):
Traceback (most recent call last):
  File "/home/vsc/LLM_TUNE/structure-stability/venv/venv-stability/lib/python3.10/site-packages/torch/utils/data/dataloader.py", line 1707, in __del__
      File "/home/vsc/LLM_TUNE/structure-stability/venv/venv-stability/lib/python3.10/site-packages/torch/utils/data/dataloader.py", line 1707, in __del__
Exception ignored in:     self._shutdown_workers()<function _MultiProcessingDataLoaderIter.__del__ at 0xfdede9f81510>    self._shutdown_workers()

self._shutdown_workers()
  File "/home/vsc/

{'loss': 0.09712736277530591,
 'mae': 1.6816662549972534,
 'rmse': 2.118018865585327,
 'euclidean': 3.319140672683716}

In [10]:
model.eval()
batch = next(iter(val_loader))
with torch.inference_mode():
    pred = model(batch['image'].to(device)).cpu()
pred_mm = pred * target_stats['std'] + target_stats['mean']
preview = torch.cat([pred_mm[:8], batch['raw_target'][:8]], dim=1)
print('columns: pred_x pred_y pred_z target_x target_y target_z')
preview

columns: pred_x pred_y pred_z target_x target_y target_z


tensor([[ 1.9952e+00, -1.4807e-01, -1.1317e+01, -1.4896e+00,  1.2216e+00,
         -1.4466e+01],
        [ 3.4050e+00, -4.1476e-03, -1.1765e+01,  1.2614e-01,  7.1402e-01,
         -1.4782e+01],
        [ 5.8783e+00, -1.8490e-01, -1.1681e+01,  1.7259e+00,  8.0181e-01,
         -1.5053e+01],
        [ 6.1675e+00, -1.8231e-01, -1.1688e+01,  1.6576e+00,  5.2803e-01,
         -1.4906e+01],
        [ 5.4273e+00, -3.9091e-01, -1.1702e+01,  1.0537e+00,  1.4307e-01,
         -1.4646e+01],
        [ 5.1273e+00, -4.1650e-01, -1.2163e+01,  6.2473e-01,  5.9345e-02,
         -1.4590e+01],
        [ 4.7825e+00, -9.0346e-02, -1.1767e+01, -3.9265e-01,  2.0627e+00,
         -1.4965e+01],
        [ 3.9478e+00, -3.4593e-01, -1.2092e+01, -9.6270e-01,  4.1000e+00,
         -1.5182e+01]])